# Chapter 3: Introduction to Text Generation
**Module 03 – Deep Learning for Text with PyTorch**

> *Instructor: Shubham Jain, Data Scientist*

## 3.1 Text Generation and NLP

Text generation models predict the **next character/word** given the previous ones.

**Key applications:**
- Chatbots
- Language translation
- Code completion
- Story/article generation

**Models used:**
- **RNN** — basic sequential model
- **LSTM** — remembers longer context
- **GRU** — efficient alternative to LSTM

Example:
```
Input:  "The cat is on the m"
Output: "The cat is on the mat"
```

## 3.2 Character-Level Text Generation

Character-level models learn to predict one character at a time — great for understanding the mechanics of text generation.

In [ ]:
import torch
import torch.nn as nn
import numpy as np

# Data preparation
data = "Hello how are you? I hope you are doing well!"
chars = list(set(data))
vocab_size = len(chars)

char_to_ix = {char: i for i, char in enumerate(sorted(chars))}
ix_to_char = {i: char for char, i in char_to_ix.items()}

print(f"Vocabulary: {''.join(sorted(chars))}")
print(f"Vocab size: {vocab_size}")

# Encode the text
encoded = [char_to_ix[c] for c in data]
print(f"Encoded first 10 chars: {encoded[:10]}")

In [ ]:
# Prepare input (all but last) and target (all but first)
inputs_idx  = torch.tensor([char_to_ix[ch] for ch in data[:-1]], dtype=torch.long).view(-1, 1)
targets_idx = torch.tensor([char_to_ix[ch] for ch in data[1:]],  dtype=torch.long)

# One-hot encode inputs
inputs_onehot = nn.functional.one_hot(inputs_idx.squeeze(), num_classes=vocab_size).float()
# Add batch dimension: (seq_len, batch=1, features)
inputs_onehot = inputs_onehot.unsqueeze(0)

print(f"Inputs shape:  {inputs_onehot.shape}")
print(f"Targets shape: {targets_idx.shape}")

## 3.3 RNN Model for Text Generation

In [ ]:
class RNNModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(RNNModel, self).__init__()
        self.hidden_size = hidden_size
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
        self.fc  = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # Initialize hidden state to zeros
        h0 = torch.zeros(1, x.size(0), self.hidden_size)
        out, _ = self.rnn(x, h0)
        # Use last timestep's output for prediction
        out = self.fc(out[:, -1, :])
        return out


model = RNNModel(
    input_size=vocab_size,
    hidden_size=32,
    output_size=vocab_size
)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

print(model)

## 3.4 Training the RNN

In [ ]:
num_epochs = 50

for epoch in range(num_epochs):
    model.train()
    outputs = model(inputs_onehot)
    loss = criterion(outputs, targets_idx)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{num_epochs} | Loss: {loss.item():.4f}")

## 3.5 Generating Text

In [ ]:
def generate_text(model, seed_char, length=50):
    """Generate text starting from a seed character."""
    model.eval()
    generated = seed_char
    current_char = seed_char

    for _ in range(length):
        # Encode current character
        idx = torch.tensor([char_to_ix[current_char]], dtype=torch.long)
        one_hot = nn.functional.one_hot(idx, num_classes=vocab_size).float().unsqueeze(0)

        # Get next character prediction
        with torch.no_grad():
            output = model(one_hot)
        probabilities = torch.softmax(output, dim=-1)
        next_idx = torch.multinomial(probabilities, 1).item()
        next_char = ix_to_char[next_idx]

        generated += next_char
        current_char = next_char

    return generated


sample = generate_text(model, seed_char='H', length=60)
print("Generated:", sample)

## Summary

| Model | Strength |
|---|---|
| RNN | Simple, fast; struggles with long context |
| LSTM | Better long-term memory |
| GRU | Efficient, often comparable to LSTM |

**Character-level generation** is conceptually simpler but word-level (or subword) is used in modern LLMs.